<a href="https://colab.research.google.com/github/timi-le/ATRX/blob/master/MScFE600_GWP1_Group14410_Replication_ECH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MScFE 600 Group Project 1
## Replication Notebook
### Group Members
- Chen Jiong Rong
- Timilehin Aderinwale Olapade
- Ikaelelo Dikhing

## 1. Objective
Replicate part of the paper using fund: ECH

## 2. Package Setup

## 3. Data Download

## 4. Data Cleaning and Preparation

## 5. Exploratory Analysis

## 6. Feature / Metric Construction

## 7. Cross-Validation

## 8. Results Table

## 9. Graphs

## 10. Brief Interpretation of Results

In [1]:
# Install the yfinance package
!pip -q install yfinance

# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Display settings
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (10, 5)

In [3]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

ticker = "ECH"
start_date = "2009-12-12"
end_date = "2020-01-01"
df = yf.download(ticker, start=start_date, end=end_date)


if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)


print(f"Data successfully downloaded for {ticker}")
print(f"Start Date in Data: {df.index.min()}")
print(f"End Date in Data: {df.index.max()}")
print(f"Total Trading Days: {len(df)}")


df['SMA_10'] = df['Close'].rolling(window=10).mean()
df['SMA_50'] = df['Close'].rolling(window=50).mean()


delta = df['Close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs = gain / loss
df['RSI'] = 100 - (100 / (1 + rs))


df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)
df.dropna(inplace=True)


features = ['Open', 'High', 'Low', 'Close', 'Volume', 'SMA_10', 'SMA_50', 'RSI']
X = df[features]
y = df['Target']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = LogisticRegression()
kf = KFold(n_splits=5, shuffle=False)
scores = cross_val_score(model, X_scaled, y, cv=kf)


print("\n--- Model Performance (Accuracy per Fold) ---")
for i, score in enumerate(scores, 1):
    print(f"Fold {i}: {score:.2%}")
print(f"Average Accuracy: {scores.mean():.2%}")


print("\n--- Last 5 Rows of Dataset ---")
print(df.tail())

/tmp/ipykernel_1747/1436160382.py:12: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed

Data successfully downloaded for ECH
Start Date in Data: 2009-12-14 00:00:00
End Date in Data: 2019-12-31 00:00:00
Total Trading Days: 2529

--- Model Performance (Accuracy per Fold) ---
Fold 1: 51.21%
Fold 2: 53.02%
Fold 3: 50.81%
Fold 4: 48.99%
Fold 5: 53.63%
Average Accuracy: 51.53%

--- Last 5 Rows of Dataset ---
Price           Close       High        Low       Open  Volume     SMA_10  \
Date                                                                        
2019-12-24  26.451025  26.624783  26.316754  26.498412  134300  26.531628   
2019-12-26  26.514212  26.569499  26.356247  26.451028  295400  26.569722   
2019-12-27  26.427326  26.577394  26.174585  26.482613  209700  26.569967   
2019-12-30  26.142994  26.395736  26.000826  26.387839  558300  26.509468   
2019-12-31  26.324650  26.569492  26.166686  26.293056  198600  26.459709   

Price          SMA_50        RSI  Target  
Date                                      
2019-12-24  26.617555  77.770788       1  
2019-12-26  

In [2]:
ticker = "ECH"
start_date = "2009-12-12"
end_date = "2020-01-01"

df = yf.download(ticker, start=start_date, end=end_date)
df.head()

/tmp/ipykernel_1747/2220398203.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,ECH,ECH,ECH,ECH,ECH
Date,,,,,
2009-12-14,36.484589,36.638447,36.056460,36.190249,212300
2009-12-15,36.357487,36.511348,36.290593,36.290593,122100
2009-12-16,36.397633,36.792314,36.324048,36.792314,201500
2009-12-17,36.263840,36.337425,35.855779,35.882538,72300
2009-12-18,36.384258,36.464532,35.788892,36.464532,104900
